# Pokémon price-history scraper

Scrapes **price over time** for one PriceCharting set (starting with **Promo**).

Every card page embeds a `VGPC.chart_data` object with ~monthly price points for
six grade tiers. We map PriceCharting's internal keys to real grade labels:

| internal key | grade |
|---|---|
| `used` | Ungraded |
| `cib` | Grade 7 |
| `new` | Grade 8 |
| `graded` | Grade 9 |
| `boxonly` | BGS 9.5 |
| `manualonly` | PSA 10 |

**Output:** `../data/pokemon_promo_history.csv` — one row per card per date, with a
`Date` and `Year` column plus a price column for each grade (in dollars).

Run order: **Config → Helpers → Phase 1 (card list) → Phase 2 (history) → Preview**.


In [1]:
# ============================================================
# Config — scrape ONE set at a time. Start with Promo.
# ============================================================
SET_SLUG = "pokemon-promo"     # PriceCharting console slug
SET_NAME = "Promo"             # label stored in the output

_stub       = SET_SLUG.replace("-", "_")
CARDS_CSV   = f"../data/{_stub}_cards.csv"      # card list: name, url, image
HISTORY_CSV = f"../data/{_stub}_history.csv"    # the new price-over-time data

# Set LIMIT to a small int (e.g. 20) to test on a few cards first; None = all.
LIMIT = None

REQUEST_DELAY = 0.4            # seconds between card-page requests (be polite)

# PriceCharting's 6 internal price series -> real grade labels
# (verified by matching the latest point of each series to the on-page prices).
GRADE_MAP = {
    "used":       "Ungraded",
    "cib":        "Grade 7",
    "new":        "Grade 8",
    "graded":     "Grade 9",
    "boxonly":    "BGS 9.5",
    "manualonly": "PSA 10",
}
GRADE_COLUMNS = list(GRADE_MAP.values())

BASE = "https://www.pricecharting.com"
UA = ("Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 "
      "(KHTML, like Gecko) Chrome/120.0 Safari/537.36")

In [2]:
# ---------- Helpers ----------
import re, json, time, csv
import datetime as dt

# Use requests if available, otherwise fall back to the stdlib (urllib).
try:
    import requests
    def _get(url):
        r = requests.get(url, headers={"User-Agent": UA}, timeout=30)
        return r.text if r.status_code == 200 else None
except ImportError:
    import urllib.request, urllib.error
    def _get(url):
        req = urllib.request.Request(url, headers={"User-Agent": UA})
        try:
            with urllib.request.urlopen(req, timeout=30) as resp:
                return resp.read().decode("utf-8", "replace")
        except urllib.error.URLError:
            return None

# Reduce a full card name down to the base species (e.g. "Charizard [1st Edition] #4" -> "Charizard")
STRIP = re.compile(
    r'\s*#\S+.*$|\b(EX|GX|VMAX|VSTAR|VMax|VStar|ex|Tag Team|Mega|Break|BREAK|'
    r'Radiant|Prism Star|Prime|Legend|SP)\b',
    re.IGNORECASE,
)

def extract_species(name):
    s = re.sub(r'\[.*?\]', '', name)      # drop [1st Edition], [Winner], ...
    s = STRIP.sub("", s)
    s = re.sub(r'\s+', ' ', s).strip().rstrip("-& ")
    return s

_CHART_RE = re.compile(r'VGPC\.chart_data\s*=\s*(\{.*?\});', re.S)

def parse_history(html):
    """Extract VGPC.chart_data -> list of per-date rows.
    Prices converted from cents to dollars; 0 (no sales that month) becomes ''."""
    m = _CHART_RE.search(html)
    if not m:
        return []
    data = json.loads(m.group(1))

    by_ts = {}                                  # timestamp_ms -> {grade: price}
    for key, label in GRADE_MAP.items():
        for ts, cents in data.get(key, []):
            if not cents:
                continue
            by_ts.setdefault(ts, {})[label] = round(cents / 100.0, 2)

    rows = []
    for ts in sorted(by_ts):
        d = dt.datetime.fromtimestamp(ts / 1000, dt.timezone.utc).date()
        row = {"Date": d.isoformat(), "Year": d.year}
        for label in GRADE_COLUMNS:
            row[label] = by_ts[ts].get(label, "")
        rows.append(row)
    return rows

def fetch(url, retries=3):
    """GET a page (no browser needed — chart_data is server-rendered)."""
    for attempt in range(retries):
        try:
            html = _get(url)
            if html:
                return html
        except Exception:
            pass
        time.sleep(1.5 * (attempt + 1))
    return None

In [3]:
# ---------- Phase 1: card list (name + page URL + image) ----------
# The console listing lazy-loads rows, so we use Playwright to scroll it all in.
import asyncio, sys
from playwright.async_api import async_playwright

async def scrape_card_list():
    url = f"{BASE}/console/{SET_SLUG}"
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page(user_agent=UA)
        print(f"Loading {url} ...")
        await page.goto(url, wait_until="networkidle", timeout=60000)

        prev = 0
        for _ in range(60):
            await page.evaluate("window.scrollTo(0, document.body.scrollHeight)")
            await asyncio.sleep(1.2)
            n = await page.locator("table#games_table tbody tr").count()
            if n == prev:
                break
            prev = n
            print(f"  loaded {n} rows...")
        print(f"Total rows: {prev}")

        cards = []
        for row in await page.locator("table#games_table tbody tr").all():
            link = row.locator("td a").first
            if await link.count() == 0:
                continue
            href = await link.get_attribute("href") or ""
            name = (await link.inner_text()).strip()
            if not href or not name:
                continue
            img_el = row.locator("td img").first
            img = ""
            if await img_el.count() > 0:
                img = await img_el.get_attribute("src") or ""
                if img.startswith("/"):
                    img = BASE + img
            cards.append({
                "Set": SET_NAME,
                "Species": extract_species(name),
                "Card Name": name,
                "Card URL": (BASE + href) if href.startswith("/") else href,
                "Image URL": img,
            })
        await browser.close()
    return cards

if "ipykernel" in sys.modules:
    import nest_asyncio
    nest_asyncio.apply()
cards = asyncio.get_event_loop().run_until_complete(scrape_card_list())

# de-dupe by URL, preserve order
seen, unique = set(), []
for c in cards:
    if c["Card URL"] not in seen:
        seen.add(c["Card URL"]); unique.append(c)
cards = unique

with open(CARDS_CSV, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=["Set", "Species", "Card Name", "Card URL", "Image URL"])
    w.writeheader(); w.writerows(cards)
print(f"Saved {len(cards)} cards -> {CARDS_CSV}")

Loading https://www.pricecharting.com/console/pokemon-promo ...
  loaded 300 rows...
  loaded 450 rows...
  loaded 600 rows...
  loaded 750 rows...
  loaded 900 rows...
  loaded 1050 rows...
  loaded 1200 rows...
  loaded 1350 rows...
  loaded 1500 rows...
  loaded 1650 rows...
  loaded 1800 rows...
  loaded 1950 rows...
  loaded 2100 rows...
  loaded 2250 rows...
  loaded 2400 rows...
  loaded 2478 rows...
Total rows: 2478
Saved 0 cards -> ../data/pokemon_promo_cards.csv


In [4]:
# ---------- Phase 2: price history per card (the timeline data) ----------
# Uses plain HTTP (fast). Writes incrementally so a long run isn't lost if interrupted.
with open(CARDS_CSV, newline="", encoding="utf-8") as f:
    cards = list(csv.DictReader(f))

todo = cards if LIMIT is None else cards[:LIMIT]
print(f"Scraping history for {len(todo)} cards (of {len(cards)})...")

fieldnames = ["Set", "Species", "Card Name", "Card URL", "Image URL", "Date", "Year"] + GRADE_COLUMNS

n_points, n_failed = 0, 0
with open(HISTORY_CSV, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    for i, card in enumerate(todo, 1):
        html = fetch(card["Card URL"])
        if not html:
            n_failed += 1
            print(f"  [{i}/{len(todo)}] FAILED: {card['Card Name']}")
            continue
        hist = parse_history(html)
        for pt in hist:
            writer.writerow({
                "Set": card["Set"], "Species": card["Species"],
                "Card Name": card["Card Name"], "Card URL": card["Card URL"],
                "Image URL": card["Image URL"], **pt,
            })
        n_points += len(hist)
        if i % 25 == 0 or i == len(todo):
            f.flush()
            print(f"  [{i}/{len(todo)}] {card['Card Name'][:40]:40} (+{len(hist)} pts | {n_points} total)")
        time.sleep(REQUEST_DELAY)

print(f"\nDone. {n_points} price points from {len(todo)-n_failed} cards "
      f"({n_failed} failed) -> {HISTORY_CSV}")

Scraping history for 0 cards (of 0)...

Done. 0 price points from 0 cards (0 failed) -> ../data/pokemon_promo_history.csv


In [5]:
# ---------- Preview ----------
import pandas as pd
h = pd.read_csv(HISTORY_CSV)
print("rows:", len(h),
      "| cards:", h["Card Name"].nunique(),
      "| dates:", h["Date"].min(), "->", h["Date"].max())
print("years:", sorted(h["Year"].unique()))
h.head(10)

rows: 0 | cards: 0 | dates: nan -> nan
years: []


,Set,Species,Card Name,Card URL,Image URL,Date,Year,Ungraded,Grade 7,Grade 8,Grade 9,BGS 9.5,PSA 10
